In [79]:
import re
import PyPDF2
import pdfplumber
from docx import Document

In [80]:
docx_file = 'Фрагмент.docx'
pdf_file = 'Фрагмент.pdf'

In [81]:
def load_docx(file_path):
    doc = Document(file_path)
    full_text = []
    for paragraph in doc.paragraphs:
        full_text.append(paragraph.text)
    return ' '.join(full_text)

In [82]:
docx_text = load_docx(docx_file)

In [83]:
def load_pdf_pypdf2(file_path):
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ''
        for page in reader.pages:
            text += page.extract_text()
    return text

def load_pdf_pdfplumber(file_path):
    with pdfplumber.open(file_path) as pdf:
        text = ''
        for page in pdf.pages:
            text += page.extract_text()
    return text

In [84]:
pdf_text_pypdf2 = load_pdf_pypdf2(pdf_file)
pdf_text_pdfplumber = load_pdf_pdfplumber(pdf_file)

2.	Предобработка текстов должна заключаться в следующем:
- Убрать все лишние символы (кроме дефиса, который нужно проверить, не находится ли он внутри слова).
- Привести все к нижнему регистру.
- Разбить весь текст на слова.

In [85]:
def preprocess_text(text):
    text = re.sub(r'[^a-zA-Zа-яА-ЯёЁ\s\-]', '', text)
    text = text.lower()
    words = text.split()
    return words

In [86]:
docx_words = preprocess_text(docx_text)
pdf_words_pypdf2 = preprocess_text(pdf_text_pypdf2)
pdf_words_pdfplumber = preprocess_text(pdf_text_pdfplumber)

3.	Посчитать:
- Общее кол-во слов.
- Кол-во уникальных слов.

In [87]:
def count_words(words):
    total_words = len(words)
    unique_words = len(set(words))
    return total_words, unique_words

In [93]:
docx_total, docx_unique = count_words(docx_words)
pdf_total_pypdf2, pdf_unique_pypdf2 = count_words(pdf_words_pypdf2)
pdf_total_pdfplumber, pdf_unique_pdfplumber = count_words(pdf_words_pdfplumber)

print(f"{docx_total=}, {docx_unique=}")
print(f"{pdf_total_pypdf2=}, {pdf_unique_pypdf2=}")
print(f"{pdf_total_pdfplumber=}, {pdf_unique_pdfplumber=}")

docx_total=3437, docx_unique=1735
pdf_total_pypdf2=3517, pdf_unique_pypdf2=1782
pdf_total_pdfplumber=3433, pdf_unique_pdfplumber=1739


4.	Выделить списки слов, состоящих из 1-2 букв.

In [89]:
def find_short_words(words):
    short_words = [word for word in words if len(word) in [1, 2]]
    return short_words

In [90]:
docx_short_words = find_short_words(docx_words)
pdf_short_words_pypdf2 = find_short_words(pdf_words_pypdf2)
pdf_short_words_pdfplumber = find_short_words(pdf_words_pdfplumber)

print(f"{docx_short_words=}")
print(f"{pdf_short_words_pypdf2=}")
print(f"{pdf_short_words_pdfplumber=}")

docx_short_words=['с', 'не', 'на', 'по', 'то', 'об', 'он', 'не', 'а', 'и', 'не', 'от', 'мы', 'с', 'и', 'до', 'в', 'мы', 'у', 'на', 'а', 'в', 'же', 'я', 'на', 'и', 'от', 'во', 'я', 'в', 'и', 'я', 'же', 'во', 'ли', 'во', 'в', 'у', 'ее', 'из', 'о', 'на', 'не', 'не', 'в', 'на', 'в', 'я', 'и', 'в', 'я', 'за', 'и', 'на', 'не', 'и', 'из', 'и', 'я', 'п', 'по', 'я', 'п', 'и', 'на', 'он', 'и', 'не', 'в', 'эх', 'а', 'не', 'я', 'на', 'на', 'на', 'с', 'и', 'же', 'в', 'в', 'от', 'с', 'ах', 'я', 'в', 'в', 'я', 'и', 'за', 'я', 'в', 'а', 'я', 'не', 'не', 'в', 'и', 'в', 'в', 'а', 'в', 'и', 'в', 'на', 'на', 'на', 'на', 'а', 'и', 'и', 'да', 'я', 'бы', 'в', 'в', 'в', 'ан', 'и', 'и', 'то', 'же', 'а', 'и', 'и', 'в', 'ни', 'на', 'и', 'на', 'я', 'за', 'но', 'и', 'с', 'и', 'на', 'по', 'эх', 'ты', 'но', 'я', 'не', 'у', 'их', 'эй', 'эй', 'и', 'эй', 'в', 'к', 'и', 'я', 'по', 'ко', 'в', 'и', 'он', 'и', 'на', 'ко', 'и', 'вы', 'я', 'я', 'уж', 'мы', 'и', 'же', 'он', 'за', 'на', 'и', 'я', 'за', 'в', 'в', 'и', 'в', 'я',

In [91]:
def find_differences(words1, words2):
    i, j = 0, 0
    len1, len2 = len(words1), len(words2)

    differences = []

    while i < len1 or j < len2:
        if i < len1 and j < len2:
            if words1[i] != words2[j]:
                differences.append((words1[i], words2[j]))
            i += 1
            j += 1
        elif i < len1:
            differences.append((words1[i], None))
            i += 1
        elif j < len2:
            differences.append((None, words2[j]))
            j += 1

    return differences

In [92]:
differences =  find_differences(pdf_words_pypdf2, pdf_words_pdfplumber)
print('pypdf2 <-> pdfplumber')
for diff in differences:
    print(f"Различие: {diff[0]} <-> {diff[1]}")

pypdf2 <-> pdfplumber
Различие: замечат <-> замечательного
Различие: ельного <-> города
Различие: города <-> грачевки
Различие: грачевки <-> а
Различие: а <-> в
Различие: в <-> два
Различие: два <-> часа
Различие: часа <-> пять
Различие: пять <-> минут
Различие: минут <-> сентября
Различие: сентября <-> того
Различие: того <-> же
Различие: же <-> -го
Различие: -го <-> незабываемого
Различие: незабываемого <-> года
Различие: года <-> я
Различие: я <-> стоял
Различие: стоял <-> на
Различие: на <-> битой
Различие: битой <-> умирающей
Различие: умирающей <-> и
Различие: и <-> смякшей
Различие: смякшей <-> от
Различие: от <-> сентябрьского
Различие: сентябрьского <-> дождика
Различие: дождика <-> траве
Различие: траве <-> во
Различие: во <-> дворе
Различие: дворе <-> мурьевской
Различие: мурьевской <-> больницы
Различие: больницы <-> стоял
Различие: стоял <-> я
Различие: я <-> в
Различие: в <-> таком
Различие: таком <-> виде
Различие: виде <-> ноги
Различие: ноги <-> окостенели
Различие: ок